# Week 17: Production Retrieval-Augmented Generation (Advanced RAG) & Vector Search
## Notebook 06: GraphRAG: Knowledge Graphs & Hierarchical Community Summarization

### 1. Conceptual Intuition & Real-World Use Case
Standard Vector RAG fails on **Global Sense-Making Queries**:
- If you ask: *"What are the top 3 overall themes across these 5,000 legal contracts?"*
- Standard vector search performs top-$k$ nearest neighbors on specific chunks, completely missing high-level cross-document patterns.

In 2024, Microsoft Research introduced **GraphRAG**:
1. Uses an LLM to extract **Entities (Nodes)** and **Relationships (Edges)** from text chunks.
2. Builds a global Knowledge Graph and detects hierarchical communities using the **Leiden Algorithm**.
3. Generates **Pre-computed Community Summaries** at multiple hierarchy levels, allowing instant answers to global queries!

---

### 2. Architecture & Pipeline
$$\text{Corpus} \xrightarrow[\text{LLM Extraction}]{\text{Entities/Edges}} \text{Knowledge Graph} \xrightarrow{\text{Leiden Algorithm}} \text{Hierarchical Clusters} \xrightarrow{\text{Map-Reduce}} \text{Global Answer}$$

In [ ]:
from collections import defaultdict
import numpy as np

# Step 1: Simulate Extracted Knowledge Graph Triplets (Entity - Relation - Entity)
triplets = [
    ("Alice", "leads_team", "Project_Falcon"),
    ("Bob", "contributes_to", "Project_Falcon"),
    ("Project_Falcon", "uses_framework", "PyTorch"),
    ("Charlie", "manages_budget", "AI_Division"),
    ("AI_Division", "funds", "Project_Falcon"),
    ("Diana", "audits_compliance", "AI_Division")
]

# Step 2: Build Adjacency and Detect Communities
entity_graph = defaultdict(set)
for u, rel, v in triplets:
    entity_graph[u].add(v)
    entity_graph[v].add(u)

# Simple Community Partition (Simulating Leiden clustering)
communities = {
    "Community_Engineering": ["Alice", "Bob", "Project_Falcon", "PyTorch"],
    "Community_Governance": ["Charlie", "Diana", "AI_Division"]
}

# Step 3: Pre-Compute Hierarchical Community Summaries
community_summaries = {
    "Community_Engineering": (
        "Focuses on technical execution of Project Falcon using the PyTorch framework, "
        "led by Alice with engineering contributions from Bob."
    ),
    "Community_Governance": (
        "Oversees financial allocation and organizational compliance for the AI Division, "
        "managed by Charlie and audited by Diana."
    )
}

# Step 4: Answer Global High-Level Query via Map-Reduce
query = "What are the core organizational pillars operating in the AI initiative?"

print("--- GraphRAG Global Query Processing ---")
print(f"Query: '{query}'\n")

# Map Phase: Collect answers from each community summary
intermediate_findings = []
for comm_id, summary in community_summaries.items():
    finding = f"[{comm_id}]: {summary}"
    intermediate_findings.append(finding)
    print("Community Report Extracted:")
    print(" ", finding)

# Reduce Phase: Synthesize final global summary
final_global_answer = (
    "The AI initiative consists of two primary pillars: "
    "1) Technical Development (Project Falcon executing ML engineering in PyTorch), and "
    "2) Governance & Finance (AI Division budgeting and regulatory compliance auditing)."
)

print("\n--- Final Synthesized Global RAG Response ---")
print(final_global_answer)

### 3. Step-by-Step Code Walkthrough
1. **Entity Extraction:** Relations across documents form a connected graph.
2. **Community Clustering:** Groups dense modules of related entities.
3. **Map-Reduce Synthesis:** Rather than retrieving 50 raw text fragments, GraphRAG evaluates concise community summaries to provide comprehensive global intelligence.

---

### 4. Industry Applications & Trade-offs
- **Intelligence Analysis & Due Diligence:** Analyzing enterprise M&A datasets or internal corporate wikis.
- **Trade-off:** High indexing cost: extracting entities and building graphs across 1,000 documents requires significant LLM token generation up front.

---

### 5. Top Interview Questions & Answers
**Q1: How does GraphRAG solve the "needle-in-a-haystack" failure of standard vector search?**
> **Answer:** Standard vector search relies on query keywords aligning with chunk embeddings. If a query asks for global themes or multi-hop relationship deductions spanning 10 different documents, no single vector embedding captures the answer. GraphRAG's hierarchical graph communities synthesize cross-document information ahead of time.

**Q2: What algorithm is used to partition knowledge graphs into communities in GraphRAG?**
> **Answer:** The **Leiden Algorithm**, an improved hierarchical clustering algorithm that guarantees well-connected communities and eliminates the disconnected sub-community flaw of the Louvain method.